In [1]:
# Run only if Ray is not installed yet
# Official Ray ML install command includes tune/train extras.

!pip install -U "ray[data,train,tune,serve]"

  Using cached tensorboardx-2.6.4-py3-none-any.whl.metadata (6.2 kB)
  Using cached pydantic_core-2.41.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 MB 2.2 MB/s  0:00:30m0:00:0100:01
Using cached pydantic_core-2.41.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 2.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 2.9 MB/s  0:00:00 eta 0:00:01
Using cached tensorboardx-2.6.4-py3-none-any.whl (87 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 2.0 MB/s  0:00:02 eta 0:00:01
  Attempting uninstall: typing-inspection
    Found existing installation: typing-inspection 0.4.1
    Uninstalling typing-inspection-0.4.1:
      Successfully uninstalled typing-i

In [2]:
from pathlib import Path
import sys
import os
import gc
import json
import math
import random
import tempfile
import importlib
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

import ray
from ray import tune
from ray.tune import Tuner
from ray.tune.tune_config import TuneConfig
from ray.tune.schedulers import PopulationBasedTraining

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
SCRIPTS_DIR = ROOT / "scripts"
RUNNER_PATH = SCRIPTS_DIR / "phase5_adaptive_wavelet_model.py"

if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Missing runner file: {RUNNER_PATH}")

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import phase5_adaptive_wavelet_model as p5
importlib.reload(p5)

SEARCH_DIR = ROOT / "results" / "pbt_search"
SEARCH_DIR.mkdir(parents=True, exist_ok=True)

print("Imported p5 from:", RUNNER_PATH)
print("DEVICE visible to notebook:", p5.DEVICE)
print("All datasets:", p5.ALL_DATASETS)
print("Search dir:", SEARCH_DIR)

Imported p5 from: /data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py
DEVICE visible to notebook: cuda
All datasets: ['etth1', 'etth2', 'ettm1', 'ettm2', 'weather', 'electricity', 'traffic', 'exchange', 'ili']
Search dir: /data/Sajjan_Singh/spml/wavelet_seq_project/results/pbt_search


In [3]:
# ============================================
# Search settings
# ============================================

SEARCH_DATASETS = list(p5.ALL_DATASETS)   # all 9 datasets, as requested

# This is a TEST search budget, not the final paper rerun.
# Each "training_iteration" = one round over all 9 datasets.
MAX_ROUNDS = 4
PERTURBATION_INTERVAL = 2

# PBT population size
# Use 4 first. Increase to 8 later if resources allow.
NUM_SAMPLES = 4

# Resource allocation per trial
# If you have only 1 GPU, trials will be time-multiplexed by Ray.
CPUS_PER_TRIAL = 4
GPUS_PER_TRIAL = 1

# To keep the first search practical across 9 datasets:
TRAIN_BATCHES_PER_ROUND = 32
VAL_BATCHES_PER_ROUND = 12

SEED = 42

print("SEARCH_DATASETS:", SEARCH_DATASETS)
print("MAX_ROUNDS:", MAX_ROUNDS)
print("NUM_SAMPLES:", NUM_SAMPLES)
print("TRAIN_BATCHES_PER_ROUND:", TRAIN_BATCHES_PER_ROUND)
print("VAL_BATCHES_PER_ROUND:", VAL_BATCHES_PER_ROUND)

SEARCH_DATASETS: ['etth1', 'etth2', 'ettm1', 'ettm2', 'weather', 'electricity', 'traffic', 'exchange', 'ili']
MAX_ROUNDS: 4
NUM_SAMPLES: 4
TRAIN_BATCHES_PER_ROUND: 32
VAL_BATCHES_PER_ROUND: 12


In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))

def resolve_seq_len_for_search(dataset_name):
    # keep this consistent with your existing "searched" convention
    seq_candidates = p5.resolve_seq_candidates(dataset_name, "searched")
    return int(seq_candidates[min(1, len(seq_candidates) - 1)])

def resolve_batch_size_for_search(dataset_name, batch_div=2):
    return max(8, p5.DEFAULT_BATCH_SIZE[dataset_name] // batch_div)

def loss_fn(pred_scaled, y_scaled, mae_mix):
    mse_part = F.mse_loss(pred_scaled, y_scaled)
    mae_part = F.l1_loss(pred_scaled, y_scaled)
    return (1.0 - mae_mix) * mse_part + mae_mix * mae_part

def build_dataset_state(dataset_name, config, restore_blob=None):
    bundle = p5.load_bundle(dataset_name, input_mode="multivariate")
    pred_len = p5.resolve_pred_len(dataset_name, "long")
    seq_len = resolve_seq_len_for_search(dataset_name)

    batch_size = resolve_batch_size_for_search(dataset_name, batch_div=int(config["batch_div"]))

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, batch_size)

    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    model = p5.AdaptiveWaveletMixer(
        input_dim=int(bundle["values_scaled"].shape[1]),
        pred_len=pred_len,
        d_model=int(config["d_model"]),
        levels=int(config["levels"]),
        wavelet_family=config["wavelet_family"],
        num_backbone_blocks=int(config["num_backbone_blocks"]),
        dropout=float(config["dropout"]),
        filter_reg_lambda=float(config["filter_reg_lambda"]),
        gate_entropy_lambda=float(config["gate_entropy_lambda"]),
    ).cpu()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(config["lr"]),
        weight_decay=float(config["weight_decay"]),
    )

    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    if restore_blob is not None:
        model.load_state_dict(restore_blob["model"])
        optimizer.load_state_dict(restore_blob["optimizer"])
        scaler.load_state_dict(restore_blob["scaler"])

    return {
        "dataset": dataset_name,
        "bundle": bundle,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "batch_size": batch_size,
        "target_mean": target_mean,
        "target_std": max(target_std, 1e-8),
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "model": model,
        "optimizer": optimizer,
        "scaler": scaler,
    }

def move_model(model, device):
    model.to(device)
    return model

def train_one_round(ds_state, config):
    model = move_model(ds_state["model"], p5.DEVICE)
    model.train()

    optimizer = ds_state["optimizer"]
    scaler = ds_state["scaler"]
    batches_done = 0
    losses = []

    for batch in ds_state["train_loader"]:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y = batch["y_scaled"].to(p5.DEVICE, non_blocking=True)

        noise_std = float(config["noise_aug_std"])
        if noise_std > 0:
            x = x + torch.randn_like(x) * noise_std

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            pred_scaled, aux = model(x)
            loss = loss_fn(pred_scaled, y, mae_mix=float(config["mae_mix"]))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        grad_clip = float(config["grad_clip"])
        if grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        scaler.step(optimizer)
        scaler.update()

        losses.append(float(loss.detach().cpu().item()))
        batches_done += 1
        if batches_done >= int(config["train_batches_per_round"]):
            break

    ds_state["model"] = move_model(model, "cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return float(np.mean(losses)) if len(losses) > 0 else np.nan

@torch.no_grad()
def eval_one_dataset(ds_state, config):
    model = move_model(ds_state["model"], p5.DEVICE)
    model.eval()

    preds_raw = []
    trues_raw = []
    val_losses = []

    batches_done = 0

    for batch in ds_state["val_loader"]:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_scaled = batch["y_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            pred_scaled, aux = model(x)
            loss = loss_fn(pred_scaled, y_scaled, mae_mix=float(config["mae_mix"]))

        pred_raw = pred_scaled.detach().cpu().numpy() * ds_state["target_std"] + ds_state["target_mean"]

        preds_raw.append(pred_raw)
        trues_raw.append(y_raw)
        val_losses.append(float(loss.detach().cpu().item()))

        batches_done += 1
        if batches_done >= int(config["val_batches_per_round"]):
            break

    preds_raw = np.concatenate(preds_raw, axis=0)
    trues_raw = np.concatenate(trues_raw, axis=0)

    raw_rmse = rmse(trues_raw.reshape(-1), preds_raw.reshape(-1))
    raw_mae = mae(trues_raw.reshape(-1), preds_raw.reshape(-1))
    norm_rmse = raw_rmse / ds_state["target_std"]

    ds_state["model"] = move_model(model, "cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "val_loss": float(np.mean(val_losses)) if len(val_losses) > 0 else np.nan,
        "val_rmse_raw": raw_rmse,
        "val_mae_raw": raw_mae,
        "val_rmse_norm": norm_rmse,
    }

def save_trial_checkpoint(round_idx, config, dataset_states, ckpt_dir):
    state = {
        "round_idx": int(round_idx),
        "config_snapshot": dict(config),
        "datasets": {},
    }

    for ds, ds_state in dataset_states.items():
        state["datasets"][ds] = {
            "model": ds_state["model"].state_dict(),
            "optimizer": ds_state["optimizer"].state_dict(),
            "scaler": ds_state["scaler"].state_dict(),
        }

    ckpt_path = Path(ckpt_dir) / "trial_state.pt"
    torch.save(state, ckpt_path)

def load_trial_checkpoint(ckpt_dir):
    ckpt_path = Path(ckpt_dir) / "trial_state.pt"
    return torch.load(ckpt_path, map_location="cpu")

In [5]:
def pbt_trainable(config):
    set_seed(int(config.get("seed", SEED)))

    # Restore if PBT resumes from checkpoint
    checkpoint = tune.get_checkpoint()
    restore_state = None
    start_round = 1

    if checkpoint:
        with checkpoint.as_directory() as checkpoint_dir:
            restore_state = load_trial_checkpoint(checkpoint_dir)
        start_round = int(restore_state["round_idx"]) + 1

    dataset_states = {}

    for ds in SEARCH_DATASETS:
        restore_blob = None
        if restore_state is not None and ds in restore_state["datasets"]:
            restore_blob = restore_state["datasets"][ds]

        dataset_states[ds] = build_dataset_state(ds, config, restore_blob=restore_blob)

    round_idx = start_round

    while True:
        result = {
            "training_iteration": round_idx,
        }

        dataset_metrics = []
        train_losses = []

        for ds in SEARCH_DATASETS:
            train_loss = train_one_round(dataset_states[ds], config)
            train_losses.append(train_loss)

            metrics = eval_one_dataset(dataset_states[ds], config)
            dataset_metrics.append(metrics)

            result[f"{ds}_val_rmse_raw"] = metrics["val_rmse_raw"]
            result[f"{ds}_val_mae_raw"] = metrics["val_mae_raw"]
            result[f"{ds}_val_rmse_norm"] = metrics["val_rmse_norm"]

        result["mean_train_loss"] = float(np.nanmean(train_losses))
        result["mean_val_rmse_raw"] = float(np.mean([m["val_rmse_raw"] for m in dataset_metrics]))
        result["mean_val_mae_raw"] = float(np.mean([m["val_mae_raw"] for m in dataset_metrics]))
        result["score"] = float(np.mean([m["val_rmse_norm"] for m in dataset_metrics]))  # minimize this

        # Always checkpoint every round for PBT
        with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
            save_trial_checkpoint(round_idx, config, dataset_states, temp_checkpoint_dir)
            ckpt = tune.Checkpoint.from_directory(temp_checkpoint_dir)
            tune.report(result, checkpoint=ckpt)

        round_idx += 1

In [6]:
# ============================================
# Search space
# Architecture is sampled once per trial
# PBT mutates only optimizer/loss/augmentation knobs
# ============================================

param_space = {
    # static architecture choices (sampled at trial start)
    "levels": tune.choice([1, 2, 3]),
    "wavelet_family": tune.choice(["Haar", "Db2", "Db4"]),
    "d_model": tune.choice([64, 128]),
    "num_backbone_blocks": tune.choice([1, 2, 3]),
    "dropout": tune.choice([0.05, 0.10, 0.15]),

    # static loader choice
    "batch_div": tune.choice([1, 2]),

    # mutable training knobs
    "lr": tune.loguniform(3e-4, 2e-3),
    "weight_decay": tune.loguniform(1e-6, 1e-3),
    "mae_mix": tune.uniform(0.0, 0.4),
    "noise_aug_std": tune.choice([0.0, 0.005, 0.01, 0.02]),
    "grad_clip": tune.choice([0.0, 0.5, 1.0, 2.0]),

    # model regularization terms are fixed per trial here
    "filter_reg_lambda": tune.choice([1e-5, 1e-4, 1e-3]),
    "gate_entropy_lambda": tune.choice([1e-5, 1e-4, 1e-3]),

    # loop settings
    "train_batches_per_round": TRAIN_BATCHES_PER_ROUND,
    "val_batches_per_round": VAL_BATCHES_PER_ROUND,
    "seed": SEED,
}

pbt_scheduler = PopulationBasedTraining(
    time_attr="training_iteration",
    metric="score",
    mode="min",
    perturbation_interval=PERTURBATION_INTERVAL,
    burn_in_period=1,
    quantile_fraction=0.25,
    resample_probability=0.25,
    perturbation_factors=(1.2, 0.8),
    synch=False,
    hyperparam_mutations={
        "lr": tune.loguniform(3e-4, 2e-3),
        "weight_decay": tune.loguniform(1e-6, 1e-3),
        "mae_mix": tune.uniform(0.0, 0.4),
        "noise_aug_std": tune.choice([0.0, 0.005, 0.01, 0.02]),
        "grad_clip": tune.choice([0.0, 0.5, 1.0, 2.0]),
    },
)

print("PBT scheduler ready.")

PBT scheduler ready.


In [ ]:
# ============================================
# ONE-CELL FIXED PBT LAUNCH + SUMMARY (device-safe optimizer state)
# Replace the previous PBT launch cell with this
# ============================================

from pathlib import Path
import os
import sys
import gc
import json
import math
import random
import tempfile
import importlib.util

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

import ray
from ray import tune
from ray.tune import Tuner
from ray.tune.tune_config import TuneConfig
from ray.tune.schedulers import PopulationBasedTraining

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project").resolve()
RUNNER_PATH = (ROOT / "scripts" / "phase5_adaptive_wavelet_model.py").resolve()
SEARCH_DIR = (ROOT / "results" / "pbt_search").resolve()
SEARCH_DIR.mkdir(parents=True, exist_ok=True)

if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Missing runner file: {RUNNER_PATH}")

SEARCH_DATASETS = ["etth1", "etth2", "ettm1", "ettm2", "weather", "electricity", "traffic", "exchange", "ili"]

MAX_ROUNDS = 4
PERTURBATION_INTERVAL = 2
NUM_SAMPLES = 4

CPUS_PER_TRIAL = 4
GPUS_PER_TRIAL = 1

TRAIN_BATCHES_PER_ROUND = 32
VAL_BATCHES_PER_ROUND = 12

SEED = 42

print("ROOT:", ROOT)
print("RUNNER_PATH:", RUNNER_PATH)
print("SEARCH_DIR:", SEARCH_DIR)
print("DATASETS:", SEARCH_DATASETS)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))

def loss_fn(pred_scaled, y_scaled, mae_mix):
    mse_part = F.mse_loss(pred_scaled, y_scaled)
    mae_part = F.l1_loss(pred_scaled, y_scaled)
    return (1.0 - mae_mix) * mse_part + mae_mix * mae_part

def import_p5_from_file():
    module_name = "phase5_adaptive_wavelet_model_worker"
    spec = importlib.util.spec_from_file_location(module_name, str(RUNNER_PATH))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def move_optimizer_state(optimizer, device):
    for state in optimizer.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(device)

def save_trial_checkpoint(round_idx, config, dataset_states, ckpt_dir):
    state = {
        "round_idx": int(round_idx),
        "config_snapshot": dict(config),
        "datasets": {},
    }
    for ds, ds_state in dataset_states.items():
        state["datasets"][ds] = {
            "model": ds_state["model"].state_dict(),
            "optimizer": ds_state["optimizer"].state_dict(),
            "scaler": ds_state["scaler"].state_dict(),
        }
    ckpt_path = Path(ckpt_dir) / "trial_state.pt"
    torch.save(state, ckpt_path)

def load_trial_checkpoint(ckpt_dir):
    ckpt_path = Path(ckpt_dir) / "trial_state.pt"
    return torch.load(ckpt_path, map_location="cpu")

def pbt_trainable(config):
    p5 = import_p5_from_file()
    set_seed(int(config.get("seed", SEED)))

    def resolve_seq_len_for_search(dataset_name):
        seq_candidates = p5.resolve_seq_candidates(dataset_name, "searched")
        return int(seq_candidates[min(1, len(seq_candidates) - 1)])

    def resolve_batch_size_for_search(dataset_name, batch_div=2):
        return max(8, p5.DEFAULT_BATCH_SIZE[dataset_name] // batch_div)

    def build_dataset_state(dataset_name, config, restore_blob=None):
        bundle = p5.load_bundle(dataset_name, input_mode="multivariate")
        pred_len = p5.resolve_pred_len(dataset_name, "long")
        seq_len = resolve_seq_len_for_search(dataset_name)
        batch_size = resolve_batch_size_for_search(dataset_name, batch_div=int(config["batch_div"]))

        train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, batch_size)

        target_idx = bundle["target_idx"]
        target_mean = float(bundle["scaler_mean"][target_idx])
        target_std = float(bundle["scaler_std"][target_idx])

        model = p5.AdaptiveWaveletMixer(
            input_dim=int(bundle["values_scaled"].shape[1]),
            pred_len=pred_len,
            d_model=int(config["d_model"]),
            levels=int(config["levels"]),
            wavelet_family=config["wavelet_family"],
            num_backbone_blocks=int(config["num_backbone_blocks"]),
            dropout=float(config["dropout"]),
            filter_reg_lambda=float(config["filter_reg_lambda"]),
            gate_entropy_lambda=float(config["gate_entropy_lambda"]),
        ).cpu()

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=float(config["lr"]),
            weight_decay=float(config["weight_decay"]),
            foreach=False,   # safer for mixed restore scenarios
        )

        scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

        if restore_blob is not None:
            model.load_state_dict(restore_blob["model"])
            optimizer.load_state_dict(restore_blob["optimizer"])
            scaler.load_state_dict(restore_blob["scaler"])
            move_optimizer_state(optimizer, "cpu")

        return {
            "dataset": dataset_name,
            "bundle": bundle,
            "seq_len": seq_len,
            "pred_len": pred_len,
            "batch_size": batch_size,
            "target_mean": target_mean,
            "target_std": max(target_std, 1e-8),
            "train_loader": train_loader,
            "val_loader": val_loader,
            "test_loader": test_loader,
            "model": model,
            "optimizer": optimizer,
            "scaler": scaler,
        }

    def move_model(model, device):
        model.to(device)
        return model

    def train_one_round(ds_state, config):
        model = move_model(ds_state["model"], p5.DEVICE)
        optimizer = ds_state["optimizer"]
        scaler = ds_state["scaler"]

        move_optimizer_state(optimizer, p5.DEVICE)

        model.train()
        batches_done = 0
        losses = []

        for batch in ds_state["train_loader"]:
            x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
            y = batch["y_scaled"].to(p5.DEVICE, non_blocking=True)

            noise_std = float(config["noise_aug_std"])
            if noise_std > 0:
                x = x + torch.randn_like(x) * noise_std

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                pred_scaled, aux = model(x)
                loss = loss_fn(pred_scaled, y, mae_mix=float(config["mae_mix"]))

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            grad_clip = float(config["grad_clip"])
            if grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            scaler.step(optimizer)
            scaler.update()

            losses.append(float(loss.detach().cpu().item()))
            batches_done += 1
            if batches_done >= int(config["train_batches_per_round"]):
                break

        ds_state["model"] = move_model(model, "cpu")
        move_optimizer_state(optimizer, "cpu")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return float(np.mean(losses)) if len(losses) > 0 else np.nan

    @torch.no_grad()
    def eval_one_dataset(ds_state, config):
        model = move_model(ds_state["model"], p5.DEVICE)
        model.eval()

        preds_raw = []
        trues_raw = []
        val_losses = []
        batches_done = 0

        for batch in ds_state["val_loader"]:
            x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
            y_scaled = batch["y_scaled"].to(p5.DEVICE, non_blocking=True)
            y_raw = batch["y_raw"].cpu().numpy()

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                pred_scaled, aux = model(x)
                loss = loss_fn(pred_scaled, y_scaled, mae_mix=float(config["mae_mix"]))

            pred_raw = pred_scaled.detach().cpu().numpy() * ds_state["target_std"] + ds_state["target_mean"]

            preds_raw.append(pred_raw)
            trues_raw.append(y_raw)
            val_losses.append(float(loss.detach().cpu().item()))

            batches_done += 1
            if batches_done >= int(config["val_batches_per_round"]):
                break

        preds_raw = np.concatenate(preds_raw, axis=0)
        trues_raw = np.concatenate(trues_raw, axis=0)

        raw_rmse = rmse(trues_raw.reshape(-1), preds_raw.reshape(-1))
        raw_mae = mae(trues_raw.reshape(-1), preds_raw.reshape(-1))
        norm_rmse = raw_rmse / ds_state["target_std"]

        ds_state["model"] = move_model(model, "cpu")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return {
            "val_loss": float(np.mean(val_losses)) if len(val_losses) > 0 else np.nan,
            "val_rmse_raw": raw_rmse,
            "val_mae_raw": raw_mae,
            "val_rmse_norm": norm_rmse,
        }

    checkpoint = tune.get_checkpoint()
    restore_state = None
    start_round = 1

    if checkpoint:
        with checkpoint.as_directory() as checkpoint_dir:
            restore_state = load_trial_checkpoint(checkpoint_dir)
        start_round = int(restore_state["round_idx"]) + 1

    dataset_states = {}
    for ds in SEARCH_DATASETS:
        restore_blob = None
        if restore_state is not None and ds in restore_state["datasets"]:
            restore_blob = restore_state["datasets"][ds]
        dataset_states[ds] = build_dataset_state(ds, config, restore_blob=restore_blob)

    round_idx = start_round

    while True:
        result = {"training_iteration": round_idx}
        dataset_metrics = []
        train_losses = []

        for ds in SEARCH_DATASETS:
            train_loss = train_one_round(dataset_states[ds], config)
            train_losses.append(train_loss)

            metrics = eval_one_dataset(dataset_states[ds], config)
            dataset_metrics.append(metrics)

            result[f"{ds}_val_rmse_raw"] = metrics["val_rmse_raw"]
            result[f"{ds}_val_mae_raw"] = metrics["val_mae_raw"]
            result[f"{ds}_val_rmse_norm"] = metrics["val_rmse_norm"]

        result["mean_train_loss"] = float(np.nanmean(train_losses))
        result["mean_val_rmse_raw"] = float(np.mean([m["val_rmse_raw"] for m in dataset_metrics]))
        result["mean_val_mae_raw"] = float(np.mean([m["val_mae_raw"] for m in dataset_metrics]))
        result["score"] = float(np.mean([m["val_rmse_norm"] for m in dataset_metrics]))

        with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
            save_trial_checkpoint(round_idx, config, dataset_states, temp_checkpoint_dir)
            ckpt = tune.Checkpoint.from_directory(temp_checkpoint_dir)
            tune.report(result, checkpoint=ckpt)

        round_idx += 1

param_space = {
    "levels": tune.choice([1, 2, 3]),
    "wavelet_family": tune.choice(["Haar", "Db2", "Db4"]),
    "d_model": tune.choice([64, 128]),
    "num_backbone_blocks": tune.choice([1, 2, 3]),
    "dropout": tune.choice([0.05, 0.10, 0.15]),

    "batch_div": tune.choice([1, 2]),

    "lr": tune.loguniform(3e-4, 2e-3),
    "weight_decay": tune.loguniform(1e-6, 1e-3),
    "mae_mix": tune.uniform(0.0, 0.4),
    "noise_aug_std": tune.choice([0.0, 0.005, 0.01, 0.02]),
    "grad_clip": tune.choice([0.0, 0.5, 1.0, 2.0]),

    "filter_reg_lambda": tune.choice([1e-5, 1e-4, 1e-3]),
    "gate_entropy_lambda": tune.choice([1e-5, 1e-4, 1e-3]),

    "train_batches_per_round": TRAIN_BATCHES_PER_ROUND,
    "val_batches_per_round": VAL_BATCHES_PER_ROUND,
    "seed": SEED,
}

pbt_scheduler = PopulationBasedTraining(
    time_attr="training_iteration",
    metric="score",
    mode="min",
    perturbation_interval=PERTURBATION_INTERVAL,
    burn_in_period=1,
    quantile_fraction=0.25,
    resample_probability=0.25,
    perturbation_factors=(1.2, 0.8),
    synch=False,
    hyperparam_mutations={
        "lr": tune.loguniform(3e-4, 2e-3),
        "weight_decay": tune.loguniform(1e-6, 1e-3),
        "mae_mix": tune.uniform(0.0, 0.4),
        "noise_aug_std": tune.choice([0.0, 0.005, 0.01, 0.02]),
        "grad_clip": tune.choice([0.0, 0.5, 1.0, 2.0]),
    },
)

if ray.is_initialized():
    ray.shutdown()

ray.init(ignore_reinit_error=True, include_dashboard=False)

trainable_with_resources = tune.with_resources(
    pbt_trainable,
    resources={"cpu": CPUS_PER_TRIAL, "gpu": GPUS_PER_TRIAL},
)

print("\nLaunching fixed PBT...")

tuner = Tuner(
    trainable_with_resources,
    param_space=param_space,
    tune_config=TuneConfig(
        scheduler=pbt_scheduler,
        num_samples=NUM_SAMPLES,
    ),
    run_config=tune.RunConfig(
        name="pbt_wavelet_search_all9_fixed",
        storage_path=str(SEARCH_DIR),
        stop={"training_iteration": MAX_ROUNDS},
        verbose=1,
    ),
)

results = tuner.fit()

results_df = results.get_dataframe()
results_csv = SEARCH_DIR / "pbt_wavelet_search_all9_fixed_results.csv"
results_df.to_csv(results_csv, index=False)

best_result = results.get_best_result(metric="score", mode="min")
best_config = best_result.config
best_metrics = best_result.metrics

best_config_path = SEARCH_DIR / "pbt_best_config_fixed.json"
with open(best_config_path, "w") as f:
    json.dump(best_config, f, indent=2)

p5_local = import_p5_from_file()
rerun_rows = []
for ds in SEARCH_DATASETS:
    rerun_rows.append({
        "dataset": ds,
        "input_mode": "multivariate",
        "horizon_mode": "long",
        "lookback_mode": "searched",
        "wavelet_family": best_config["wavelet_family"],
        "levels": int(best_config["levels"]),
        "d_model": int(best_config["d_model"]),
        "num_backbone_blocks": int(best_config["num_backbone_blocks"]),
        "dropout": float(best_config["dropout"]),
        "filter_reg_lambda": float(best_config["filter_reg_lambda"]),
        "gate_entropy_lambda": float(best_config["gate_entropy_lambda"]),
        "batch_size": p5_local.DEFAULT_BATCH_SIZE[ds],
        "epochs": p5_local.DEFAULT_EPOCHS[ds],
    })

rerun_df = pd.DataFrame(rerun_rows)
rerun_csv = SEARCH_DIR / "pbt_bestcfg_full_rerun_grid_fixed.csv"
rerun_df.to_csv(rerun_csv, index=False)

print("\nSaved search table:", results_csv)
print("Saved best config:", best_config_path)
print("Saved full rerun grid:", rerun_csv)

print("\nBEST CONFIG:")
print(json.dumps(best_config, indent=2))

print("\nBEST METRICS:")
for k in ["score", "mean_val_rmse_raw", "mean_val_mae_raw", "training_iteration"]:
    if k in best_metrics:
        print(f"{k}: {best_metrics[k]}")

display(results_df.sort_values("score").head(10))
display(rerun_df)

(pbt_trainable pid=2661142) /tmp/ipykernel_2602611/297313983.py:157: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
(pbt_trainable pid=2661142) /tmp/ipykernel_2602611/297313983.py:203: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
(pbt_trainable pid=2661142) /tmp/ipykernel_2602611/297313983.py:243: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
(pbt_trainable pid=2661142) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/data/Sajjan_Singh/spml/wavelet_seq_project/results/pbt_search/pbt_wavelet_search_all9/pbt_trainable_ecd46_00000_0_batch_div=1,d_model=128,dropout=0.0500,filter_reg_lambda=0.0010,gate_entropy_lambda=0.0010,grad_clip=0_2026-03-27_18-08-58/checkpoint_000000)
(pbt_trainable pid=2661142) Checkpoint successfully created at: C


PBT finished. Building summary...

Saved search table: /data/Sajjan_Singh/spml/wavelet_seq_project/results/pbt_search/pbt_wavelet_search_all9_results.csv
Saved best config: /data/Sajjan_Singh/spml/wavelet_seq_project/results/pbt_search/pbt_best_config.json
Saved full rerun grid: /data/Sajjan_Singh/spml/wavelet_seq_project/results/pbt_search/pbt_bestcfg_full_rerun_grid.csv

BEST CONFIG:
{
  "levels": 1,
  "wavelet_family": "Haar",
  "d_model": 64,
  "num_backbone_blocks": 3,
  "dropout": 0.05,
  "batch_div": 1,
  "lr": 0.0015894996902689061,
  "weight_decay": 0.00029494600035531374,
  "mae_mix": 0.0934926110057051,
  "noise_aug_std": 0.01,
  "grad_clip": 0.0,
  "filter_reg_lambda": 1e-05,
  "gate_entropy_lambda": 0.001,
  "train_batches_per_round": 32,
  "val_batches_per_round": 12,
  "seed": 42
}

BEST METRICS:
score: 0.4849186544496461
mean_val_rmse_raw: 13612.510065181252
mean_val_mae_raw: 12300.400960957513
training_iteration: 2


,training_iteration,etth1_val_rmse_raw,etth1_val_mae_raw,etth1_val_rmse_norm,etth2_val_rmse_raw,etth2_val_mae_raw,etth2_val_rmse_norm,ettm1_val_rmse_raw,ettm1_val_mae_raw,ettm1_val_rmse_norm,...,config/weight_decay,config/mae_mix,config/noise_aug_std,config/grad_clip,config/filter_reg_lambda,config/gate_entropy_lambda,config/train_batches_per_round,config/val_batches_per_round,config/seed,logdir
3,2,4.069543,3.281472,0.443475,7.791622,6.419270,0.672578,2.263617,1.748454,0.246769,...,0.000295,0.093493,0.010,0.0,0.00001,0.00100,32,12,42,ecd46_00003
0,2,3.957207,3.196877,0.431233,8.246069,6.807683,0.711806,2.430043,1.931893,0.264912,...,0.000091,0.255348,0.005,0.0,0.00100,0.00100,32,12,42,ecd46_00000
2,2,4.148946,3.370168,0.452128,8.988963,7.487634,0.775933,2.777621,2.286442,0.302803,...,0.000001,0.199150,0.005,0.5,0.00001,0.00001,32,12,42,ecd46_00002
1,2,3.799617,3.038718,0.414060,7.444991,6.103555,0.642656,2.849057,2.342446,0.310590,...,0.000158,0.212291,0.010,2.0,0.00001,0.00100,32,12,42,ecd46_00001


,dataset,input_mode,horizon_mode,lookback_mode,wavelet_family,levels,d_model,num_backbone_blocks,dropout,filter_reg_lambda,gate_entropy_lambda,batch_size,epochs
0,etth1,multivariate,long,searched,Haar,1,64,3,0.05,0.00001,0.001,256,30
1,etth2,multivariate,long,searched,Haar,1,64,3,0.05,0.00001,0.001,256,30
2,ettm1,multivariate,long,searched,Haar,1,64,3,0.05,0.00001,0.001,256,30
3,ettm2,multivariate,long,searched,Haar,1,64,3,0.05,0.00001,0.001,256,30
4,weather,multivariate,long,searched,Haar,1,64,3,0.05,0.00001,0.001,192,30
5,electricity,multivariate,long,searched,Haar,1,64,3,0.05,0.00001,0.001,128,30
6,traffic,multivariate,long,searched,Haar,1,64,3,0.05,0.00001,0.001,96,30
7,exchange,multivariate,long,searched,Haar,1,64,3,0.05,0.00001,0.001,256,30
8,ili,multivariate,long,searched,Haar,1,64,3,0.05,0.00001,0.001,64,35
